# 05 · Validation Plan — EMSA / fluorescence-anisotropy + scrambled-NA controls

**Standard slot:** *validation plan.* **For Project 23 this means:** turn the top specific candidates
into a **costed, controlled wet-lab plan** — an **EMSA (gel-shift)** and/or **fluorescence-anisotropy**
binding assay vs the target nucleic acid, the mandatory controls (**scrambled-NA negative**,
catalytic/interface dead-mutant of your own design, unrelated protein), an expression strategy, and the
**CRISPR-modulator** stretch (D4/D5).

A design that passes every filter is a **hypothesis** — an EMSA / anisotropy titration with a
**scrambled-NA control** is what tests both *binding* and *specificity*. Needs
`results/top_candidates.csv` (notebook 04). **No fabricated K_D anywhere** — report measured numbers
only after you measure them.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Draft the experimental validation plan

Generate a plan card from the top candidates: assays, controls, expression, timeline, costed reagents.
Fill the `<...>` from your own numbers; this is the deliverable other people will actually read. The
non-negotiables for protein–NA: a **scrambled-NA control** and a **dead-mutant control** (your own
design with its interface residues mutated), because specificity is the whole game.

In [ ]:
import pandas as pd, os

top = pd.read_csv("results/top_candidates.csv") if os.path.exists("results/top_candidates.csv") else pd.DataFrame()
n_top = len(top)
by_tool = top.groupby("seq_tool").size().to_dict() if n_top else {}
motif = str(top["target_motif"].iloc[0]) if (n_top and "target_motif" in top.columns) else "<your motif>"

plan = f"""# Protein-NA Binder Validation Plan (Project 23 - by <your name>, <date>)

## Candidates
Top {n_top} confident-AND-specific candidates carried forward ({by_tool}); see results/top_candidates.csv.
Target nucleic acid: {motif} (na_type from your design). EVERY in-silico number is a HYPOTHESIS until
measured - pae_interaction is confidence, specificity_score is a computational proxy, NEITHER is a K_D.

## Expression / reagents
- Protein binders: E. coli BL21(DE3), His-tagged, 16-18 C overnight; IMAC + SEC. Small (40-90 aa) -> high yield expected.
- Target nucleic acid: synthesize the {motif} oligo (DNA: HPLC-purified duplex; RNA: in-vitro transcribed
  or synthesized, with a 5' fluorophore (FAM/Cy5) for anisotropy). Order a SCRAMBLED-NA oligo of the SAME
  length/base-composition as the specificity control.

## Assays (go/no-go -> binding -> specificity)
1. Go/no-go: express -> SDS-PAGE -> SEC (monodisperse? not aggregated?).
2. Binding: EMSA (gel-shift) titration of protein vs labeled target NA -> apparent affinity from the shift;
   AND/OR fluorescence anisotropy/polarization titration (labeled NA, protein dilution series) -> apparent K_D.
3. SPECIFICITY (the point): repeat the SAME titration against the SCRAMBLED-NA control. A specific binder
   shifts/binds the intended motif at much lower protein concentration than the scramble. Report the RATIO,
   not just the intended-motif number.
4. Stability: DSF (Tm) of the protein. Deep (optional): co-crystal / cryo-EM of the protein-NA complex;
   competition with a known motif-binding protein.

## Controls (MANDATORY)
- Negative (scrambled-NA): the SAME assay vs a scrambled motif (same composition) -> a specific binder
  should bind it MUCH more weakly. This is the cleanest specificity control and is REQUIRED.
- Negative (dead-mutant): YOUR OWN top design with its predicted NA-interface residues mutated (e.g. the
  base-reading residues -> Ala) -> must LOSE binding to the intended motif.
- Negative (unrelated protein): an unrelated protein of similar size/charge -> should not shift the NA.
- Positive: a known binder of {motif} (the natural protein / a published designed binder, if available)
  to confirm the labeled NA reagent and the assay are working.

## Realistic expectations
Protein-NA design is NEWER and HARDER than protein-protein; SEQUENCE SPECIFICITY is the main failure mode
(designs grip the generic phosphate backbone, not the bases). Expect many in-silico hits to bind
non-specifically or not at all. Report the experimental hit rate AND the specificity ratio honestly.
Do NOT imply a working binder or fabricate a K_D.

## Timeline + costed reagents (fill in)
- Gene synthesis ({n_top} binders + dead-mutant negatives): $<...>, <...> weeks (IGSC-screened provider).
- Labeled target NA + scrambled-NA control oligos (+ unlabeled for EMSA): $<...>.
- Anisotropy plate reader / EMSA gel time + a positive-control reagent: $<...>.
- Personnel/instrument time: <...> weeks.

## Responsible research
Designed nucleic-acid-binding proteins for gene-editing modulation / RNA-targeting therapeutics /
synthetic transcription factors (therapeutic / basic-science; low dual-use). Default neutralizing/
therapeutic framing. Gene synthesis via a biosecurity-screening provider; wet lab under institutional
biosafety/ethics approval.
"""
os.makedirs("results", exist_ok=True)
open("results/validation_plan.md", "w").write(plan)
print("wrote results/validation_plan.md - fill the <...> placeholders from your numbers.")
print(plan[:700], "...")

## 2 · Build the scrambled-NA + dead-mutant negative controls

Two specificity controls, generated alongside the real designs so the EMSA/anisotropy comparison is
airtight:
1. **Scrambled-NA**: the same motif with its bases shuffled (same composition) — order this oligo and
   run the *same* assay; a specific binder should bind it much more weakly.
2. **Dead-mutant**: your own top design with its predicted NA-interface residues mutated — it should
   lose binding to the intended motif. Here we scaffold a sequence-level mutant deterministically; on
   Colab, mutate the *predicted interface* residues specifically.

In [ ]:
import random
import na_binder_tools as nbt   # nbt._hashints gives a DETERMINISTIC seed (Python's hash() is salted)

MOTIF = str(top["target_motif"].iloc[0]) if (n_top and "target_motif" in top.columns) else "TGACGTCA"
NA_TYPE = str(top["na_type"].iloc[0]) if (n_top and "na_type" in top.columns) else "DNA"

# (1) Scrambled-NA control oligo (order this; same composition, shuffled order).
scrambled_na = nbt.scramble_motif(nbt.normalize_motif(MOTIF, NA_TYPE), seed=0)
print(f"scrambled-NA control oligo: {MOTIF} -> {scrambled_na}  (order BOTH; run the SAME assay)")

# (2) Dead-mutant protein controls (mutate a fraction of residues as a NEGATIVE-CONTROL stand-in).
def dead_mutant(seq, frac=0.4, seed=0):
    """Deterministically mutate a fraction of residues -> Ala/Gly as a NEGATIVE-CONTROL stand-in.
    On Colab, mutate the PREDICTED NA-interface residues specifically (the base-reading positions)."""
    rng = random.Random(seed)
    seq = list(seq); idx = list(range(len(seq))); rng.shuffle(idx)
    for i in idx[:max(1, int(len(seq) * frac))]:
        seq[i] = "A" if rng.random() < 0.5 else "G"
    return "".join(seq)

negs = []
if n_top and "sequence" in top.columns:
    for _, r in top.iterrows():
        s = str(r.get("sequence", ""))
        if s and set(s) <= set("ACDEFGHIKLMNPQRSTVWY"):
            negs.append(dict(design_id=str(r["design_id"]) + "_DEADMUT",
                             parent=r["design_id"], seq_tool=r.get("seq_tool"),
                             sequence=dead_mutant(s, seed=nbt._hashints(r["design_id"]) % 10**6),
                             role="NA-interface dead-mutant negative control"))
    pd.DataFrame(negs).to_csv("results/negative_controls.csv", index=False)
    print(f"wrote results/negative_controls.csv: {len(negs)} dead-mutant negatives (+ the scrambled-NA oligo above)")
else:
    print("Run notebook 04 first to produce results/top_candidates.csv with sequences.")

## 3 · (Stretch) CRISPR-modulator validation `[stretch]`

If you took the CRISPR-modulator extension (bind a Cas surface to tune/block editing), the validation is
a **protein–protein** plan (SPR/BLI vs the Cas target + a *functional editing assay*: does the modulator
reduce/redirect Cas cleavage in vitro or in cells?), with the same control discipline (scrambled-interface
negative, unrelated protein). Frame it as editing **control/safety** (an anti-CRISPR-like modulator),
per Responsible Research — never as a tool to defeat safeguards for harm.

In [ ]:
# Scaffold ONLY. CRISPR-modulator validation = SPR/BLI vs the Cas target + an in-vitro/cell EDITING assay
#   (does the modulator reduce/redirect Cas cleavage?), with scrambled-interface + unrelated-protein controls.
#   Reuse the Project 06 SPR/BLI plan structure; keep the framing therapeutic/safety (editing CONTROL).
print("CRISPR-modulator validation is a STRETCH scaffold: SPR/BLI + a functional editing-control assay.")
print("Frame as safer/controllable editing (anti-CRISPR-like modulator) - never to defeat safeguards for harm.")

## D4 / D5 checklist
- [ ] `results/validation_plan.md` completed: **EMSA / fluorescence-anisotropy** titration + **scrambled-NA** repeat, expression, timeline, costed reagents.
- [ ] Controls specified: **scrambled-NA** negative (required), **dead-mutant** negative (`results/negative_controls.csv`), unrelated-protein negative, positive (known motif binder).
- [ ] Specificity reported as a **ratio** (intended vs scrambled), not just the intended-motif number; **no fabricated K_D**.
- [ ] (Stretch) CRISPR-modulator validation scaffolded as a protein–protein + functional-editing plan, framed as editing control/safety.
- [ ] Honest framing: every design is a hypothesis until EMSA/anisotropy; protein–NA specificity is the main failure mode; report the experimental hit rate.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — a complete protein–NA-binder DBTL turn, honestly reported, with specificity at its center.